In [1]:
import pandas as pd
from sklearn.svm import SVR
import joblib

# --- 1. Load Your Dataset ---
data_filename = 'multi_view_features_all_side.csv'
print(f"Loading data from '{data_filename}'...")
df = pd.read_csv(data_filename)

# --- 2. Prepare Data (X and y) ---
# As requested:
# y (target) is 'height_in_cm'
# X (features) is everything else, dropping id, gender, height, and weight

print("Preparing features (X) and target (y)...")
# Define the target variable
y = df['height_in_cm']

# Define the features by dropping the non-feature and target columns
# We drop: 'child_id', 'gender', 'height_in_cm' (target), and 'weight_in_grams'
X = df.drop(columns=['child_id', 'gender', 'height_in_cm', 'weight_in_grams'])

print(f"Data ready: X shape={X.shape}, y shape={y.shape}")
print(f"Features being used: {list(X.columns)}")

# --- 3. Define Your Best Hyperparameters ---
# !!! IMPORTANT: REPLACE these placeholder values with your *actual* best values from Optuna.
your_best_params = {
    'kernel': 'linear',      # e.g., 'rbf', 'linear', 'poly'
    'C': 0.007752522403517577,            # The C value you found
    'epsilon': 0.1594593749092818,         # The epsilon value you found
    'gamma': 'auto'      # e.g., 'scale', 'auto', or a float value like 0.01
}
print(f"Using best hyperparameters: {your_best_params}")

# --- 4. Create and Train the Model ---
print("Initializing SVR model with best parameters...")
# Create the SVR model instance using your best parameters
final_svr_model = SVR(**your_best_params)

print("Training the model on the full dataset...")
# Fit the model on your entire dataset
final_svr_model.fit(X, y)

print("Model training complete.")

# --- (Optional) How to Load and Use It Later ---
# print("\n--- Loading model for a test prediction ---")
# loaded_model = joblib.load(model_filename)
#
# # Get the first row of features to test
# test_sample = X.iloc[0:1] 
#
# prediction = loaded_model.predict(test_sample)
# actual = y.iloc[0]
#
# print(f"Test prediction on first sample: {prediction[0]:.2f}")
# print(f"Actual height for first sample: {actual}")

Loading data from 'multi_view_features_all_side.csv'...
Preparing features (X) and target (y)...
Data ready: X shape=(102, 18), y shape=(102,)
Features being used: ['age_in_months', 'max_torso_length', 'avg_torso_length', 'max_leg_length', 'avg_leg_length', 'max_arm_length', 'avg_arm_length', 'max_head_to_shoulder', 'avg_head_to_shoulder', 'max_shoulder_width', 'avg_shoulder_width', 'max_hip_width', 'avg_hip_width', 'max_head_width', 'avg_head_width', 'hip_to_shoulder', 'torso_to_leg', 'arm_to_torso']
Using best hyperparameters: {'kernel': 'linear', 'C': 0.007752522403517577, 'epsilon': 0.1594593749092818, 'gamma': 'auto'}
Initializing SVR model with best parameters...
Training the model on the full dataset...
Model training complete.


In [ ]:
# --- 5. Save the Trained Model ---
model_filename = 'final_svr_height_predictor.pkl'
joblib.dump(final_svr_model, model_filename)

print(f"\n✅ Model successfully trained and saved to: '{model_filename}'")

# 

In [4]:
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# --- 1. Load Your Dataset ---
data_filename = 'multi_view_features_all_side.csv'
print(f"Loading data from '{data_filename}'...")
df = pd.read_csv(data_filename)

# --- 2. Prepare Data (X and y) ---
print("Preparing features (X) and target (y)...")
# Define the target variable
y = df['height_in_cm']

# Define the features by dropping the non-feature and target columns
X = df.drop(columns=['child_id', 'gender', 'height_in_cm', 'weight_in_grams'])

# --- 3. Split the Data for Testing ---
# As requested: 80% train, 20% test, random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data split complete:")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Testing samples:  {len(X_test)}")

# --- 4. Define Your Best Hyperparameters ---
# !!! IMPORTANT: REPLACE these placeholder values with your *actual* best values from Optuna.
your_best_params = {
    'kernel': 'linear',      # e.g., 'rbf', 'linear', 'poly'
    'C': 0.007752522403517577,            # The C value you found
    'epsilon': 0.1594593749092818,         # The epsilon value you found
    'gamma': 'auto'      # e.g., 'scale', 'auto', or a float value like 0.01
}
print(f"Using hyperparameters: {your_best_params}")

# --- 5. Train the Model on the *Training Set* ---
print("\nTraining model on the 80% training split...")
svr_model = SVR(**your_best_params)
svr_model.fit(X_train, y_train)
print("Training on split complete.")

# --- 6. Test the Model and Get MAE ---
print("Testing model on the 20% test split...")
y_pred = svr_model.predict(X_test)

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*30)
print(f"TESTING COMPLETE")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"(This means, on average, the model's height prediction was off by {mae:.4f} cm on the unseen test data)")
print("="*30 + "\n")

# --- 7. Re-train Final Model on ALL Data ---
print("Testing is done. Now training the *final* model on 100% of the data...")
# We use the full X and y datasets now
final_model = SVR(**your_best_params)
final_model.fit(X, y) 
print("Final model training complete.")

# --- 8. Save the Final Model ---
model_filename = 'final_svr_height_predictor.pkl'
joblib.dump(final_model, model_filename)

print(f"\n✅ Final model successfully trained on all data and saved to: '{model_filename}'")

Loading data from 'multi_view_features_all_side.csv'...
Preparing features (X) and target (y)...
Data split complete:
  - Training samples: 81
  - Testing samples:  21
Using hyperparameters: {'kernel': 'linear', 'C': 0.007752522403517577, 'epsilon': 0.1594593749092818, 'gamma': 'auto'}

Training model on the 80% training split...
Training on split complete.
Testing model on the 20% test split...

TESTING COMPLETE
Mean Absolute Error (MAE): 5.9514
(This means, on average, the model's height prediction was off by 5.9514 cm on the unseen test data)

Testing is done. Now training the *final* model on 100% of the data...
Final model training complete.

✅ Final model successfully trained on all data and saved to: 'final_svr_height_predictor.pkl'


In [3]:
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder # LabelEncoder is added
import joblib

# --- 1. Load Your Dataset ---
data_filename = 'multi_view_features_all_side.csv'
print(f"Loading data from '{data_filename}'...")
df = pd.read_csv(data_filename)

# --- 1.5. Label Encode Gender ---
print("Label Encoding 'gender' column...")
le = LabelEncoder()
# Apply Label Encoding to convert 'f' and 'm' to 0 and 1
df['gender'] = le.fit_transform(df['gender'])

# --- 2. Prepare Data (X and y) ---
print("Preparing features (X) and target (y)...")
# Define the target variable
y = df['height_in_cm']

# Define the features. 'gender' is now numeric and is kept.
X = df.drop(columns=['child_id', 'height_in_cm', 'weight_in_grams'])

# Safety check: Ensure all columns are numeric for SVR
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean()) # Impute any NaNs with the mean

# --- 3. Split the Data for Testing ---
# 80% train, 20% test, random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data split complete: Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# --- 4. Define Your Best Hyperparameters ---
your_best_params = {
    'kernel': 'linear',
    'C': 0.007752522403517577,
    'epsilon': 0.1594593749092818,
    'gamma': 'auto'
}
print(f"Using hyperparameters: {your_best_params}")

# --- 5. Train the Model on the *Training Set* ---
print("\nTraining model on the 80% training split...")
svr_model = SVR(**your_best_params)
svr_model.fit(X_train, y_train)
print("Training on split complete.")

# --- 6. Test the Model and Get MAE ---
print("Testing model on the 20% test split...")
y_pred = svr_model.predict(X_test)

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*30)
print(f"TESTING COMPLETE")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print("="*30 + "\n")

# --- 7. Re-train Final Model on ALL Data ---
print("Now training the *final* model on 100% of the data...")
final_model = SVR(**your_best_params)
final_model.fit(X, y)
print("Final model training complete.")

# --- 8. Save the Final Model ---
model_filename = 'final_svr_height_predictor_with_gender.pkl'
joblib.dump(final_model, model_filename)

print(f"\n✅ Final model successfully trained on all data and saved to: '{model_filename}'")

Loading data from 'multi_view_features_all_side.csv'...
Label Encoding 'gender' column...
Preparing features (X) and target (y)...
Data split complete: Training samples: 81, Testing samples: 21
Using hyperparameters: {'kernel': 'linear', 'C': 0.007752522403517577, 'epsilon': 0.1594593749092818, 'gamma': 'auto'}

Training model on the 80% training split...
Training on split complete.
Testing model on the 20% test split...

TESTING COMPLETE
Mean Absolute Error (MAE): 5.9545

Now training the *final* model on 100% of the data...
Final model training complete.

✅ Final model successfully trained on all data and saved to: 'final_svr_height_predictor_with_gender.pkl'
